# Transaction Foundation Model on Ray — Part 1: Setup

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## What we're building

Transaction foundation models are the latest generation of transformer models — like LLMs, but instead of language, they are trained on financial transactions and similar timeseries record data. This lets them recognize patterns, like fraud, that traditional ML techniques can't detect. Stripe, Visa, Mastercard, Nubank, and Revolut all run models like this in production.

This series of notebooks shows you how to build your own transaction foundation model, with performance and scalability that surpass [NVIDIA's blueprint](https://github.com/NVIDIA-AI-Blueprints/transaction-foundation-model), using an identical approach but with Ray achieving a scale that the NVIDIA blueprint cant reach. Here, we reuse NVIDIA's tokenizer, train their exact architecture with their recipe, and score fraud with their same evaluation method. So its an apples-to-apples comparison. Ray is the only difference. Along the way we maximize our GPUs, solve bottlenecks through independently scalable CPUs, and save money using a fault-tollerant cluster of AWS Spot instances. 

This is what a production workload looks like. In comparison, NVIDIA's repo runs every stage on a single node, and it never actually trains the foundation model — its tains a small 30 steps demonstration, and then downloads the real weights. This template runs the actual training: ~16,000 steps, or 8 epochs over all 19.5 million training transactions, about two hours on eight GPUs. Dont worry, with the exact same set of notebooks, you can scale down just as easily as you scale up - see our 'mini.yaml' config file for a small cpu-only configuration of this exact same code.

## How a foundation model catches fraud

The foundation model never sees a fraud label. It pretrains on millions of unlabeled transaction sequences, learning to predict the next transaction the same way an LLM learns to predict the next word. Once trained, it can read any transaction and output an **embedding**: a vector of 512 numbers that captures what the model knows about that transaction. Similar transactions get similar embeddings.

The fraud detector itself is XGBoost — the same classifier most fraud teams run today. The foundation model changes what you feed it. NVIDIA's blueprint scores the classifier on three feature sets, and we do the same:

- **raw** — the 13 fields a fraud team already has: amount, merchant, location, time, and so on. This is the baseline to beat.
- **fm** — the foundation-model embedding alone.
- **fusion** — the raw fields plus the embedding. This is the one that matters in production: does the foundation model catch fraud that the raw fields miss?

## Model performance

We score with NVIDIA's own evaluation: average precision on their 100K-transaction test set.

| feature set | NVIDIA blueprint | this template |
|---|---|---|
| **fm** | 0.0123 | **0.04–0.06 — 3–5× better** |
| **fusion** | 0.1755 (single draw) | typical 0.136, **best draw 0.284** |
| raw (the control) | 0.1238 | 0.1238 — matched exactly |

Our foundation model beats theirs by 3–5×. Our fusion beats their fusion. The raw row is the control: it uses their exact features and their exact recipe, so it has to match — and it does, to the fourth decimal. That match proves the two wins come from the model, not from a difference in method.

One caution on reading these numbers. The test set contains only ~112 frauds, so any single score is noisy. NVIDIA published one fusion number; we report the distribution of ours. Their 0.1755 lands inside it, and we clear it in about one draw in six. The fm score also moves between reruns, so we quote the range we observed instead of a single point. Part 6 covers both in detail.

## Scalability

NVIDIA's pipeline runs every stage on one machine, and the one stage that actually needs more than one machine — pretraining — their repo skips entirely. This template distributes every stage on the hardware it needs. Data preparation runs across autoscaling CPU workers with Ray Data. Pretraining runs across eight GPUs with Ray Train. Embedding extraction streams between the two: CPU workers prepare transactions while GPU workers run the model, and each pool scales on its own, so GPUs never sit idle waiting on data work.

Distributing a pipeline can silently change its results. We verified ours didn't. Every distributed stage produces output identical to NVIDIA's single-machine version: the same 19.5 million rows in the same order, a bit-for-bit identical training corpus, byte-equal labels and features. The verification scripts are in `scripts/verify_*.py`, and the matched raw baseline above is the bottom line.

The cluster doesn't sit around either. Workers come up when a stage requests them (2–3 minutes in our runs) and scale back to zero when it finishes, so the GPUs exist for roughly the two hours that use them. NVIDIA's single A100-class node pays for its GPU the whole time, whether it's training, preparing data, or idle.

## Architecture

```
                     ┌──────────────────────── Anyscale ────────────────────────┐
 IBM TabFormer ────► │ [Ray Data]   temporal split + corpus (CPU workers)       │
 (24M txns)          │ [Ray Train]  next-token pretraining, 8 GPUs (DDP)        │
                     │ [Ray Data]   embeddings: CPU tokenize → GPU actors       │
                     │ [XGBoost]    downstream fraud: raw vs fm vs fusion (GPU) │
                     │ [Ray Serve]  online embedding + fraud score              │
                     └──────────────────────────────────────────────────────────┘
```

Every stage is the **same code** from a single machine to a multi-node cluster — you change `ScalingConfig`, not your program. One `SCALE` variable at the top of each notebook selects a config file under `configs/`, and diffing two of those files shows the complete difference between two scales.

## The series

We build the foundation model end-to-end, one stage per notebook:

| # | Notebook | Runs on |
|---|----------|---------|
| **1** | **Setup** *(this notebook)* | — |
| 2 | Load & explore the data | Ray Data, CPU workers |
| 3 | Tokenize — build the pretrain corpus | Ray Data, CPU workers |
| 4 | Pretrain — next-token prediction | Ray Train, 8 GPUs |
| 5 | Batch embedding extraction | Ray Data, CPU → GPU actors |
| 6 | Downstream fraud — raw vs FM vs fusion | XGBoost on GPU |
| 7 | Online serving | Ray Serve |
| 8 | Run the whole pipeline as a job | Anyscale Jobs |
| 9 | Scaling up — the bottlenecks you'll hit | All of the above |

## What a full run needs

`mini` proves the plumbing on CPU in minutes. The results above come from `SCALE = "full"`, which needs:

- the IBM TabFormer dataset — a one-time ~2.3 GB download that Part 2 handles,
- GPU workers — 8×A10G for the ~2h pretraining and for embedding extraction; the downstream XGBoost also needs a GPU (Part 6 explains why the result depends on it),
- a CPU worker group in the cluster config, so the CPU-heavy data stages don't pull up GPU nodes just for their CPUs (Part 9 shows that failure mode).

Nothing else changes between the two scales. Same notebooks, same code, different config file.

---

## Step 1: Install dependencies

The cell below installs the template's dependencies and registers them on every cluster node, so the same imports resolve on workers as well as the head node. Two pins matter. `xgboost` is 3.2.0 because the downstream fraud result is sensitive to an early-stopping behavior that changed in 3.3. RAPIDS (`cudf`) is NVIDIA's GPU tokenizer, kept as the reference implementation — the notebooks run a CPU implementation we verified byte-identical to it, which is why `mini` needs no GPU at all.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [1]:
!pip install -q -r requirements.txt

Successfully registered `ray, torch` and 13 other packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_g54aiirwj1s8t9ktgzikqur41k/prj_f1j47h9srml4cyg962id75ms2e/workspaces/expwrk_78mtwtucrd61tjxf851krarzwr?workspace-tab=dependencies


## Step 2: Attach to the cluster

In an Anyscale Workspace, Ray is already running — `ray.init()` **attaches** to the workspace's cluster rather than starting one. `working_dir` ships this template's `src/` package to every worker, so the notebook and remote tasks/actors all import the same code.

In [2]:
import sys, os
import logging


# Make the template's `src/` package importable from the notebook.
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray

# In an Anyscale Workspace, Ray is already running — this attaches to it.
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.utils import print_cluster_resources
print_cluster_resources()

Ray cluster resources:
  anyscale/cpu_only:true 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 1.0
  anyscale/region:us-west-2 1.0
  memory               34359738368.0
  object_store_memory  9599240601.0

Cluster nodes: 1
  10.0.158.37          alive=True  anyscale/cpu_only:true=1.0, anyscale/region:us-west-2=1.0, anyscale/provider:aws=1.0, memory=34359738368.0, object_store_memory=9599240601.0, anyscale/node-group:head=1.0


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


You should see the head node and its resources. On a fresh or idle cluster the head node intentionally schedules no work — GPU and CPU worker nodes autoscale up when later stages request them, and scale back down when idle. You don't manage that; Ray does.

That's the whole setup: dependencies registered, cluster attached. Every notebook from here starts the same way and picks up where the previous one left off, using artifacts written to shared storage.

---

## Next

**Part 2 — Load & explore the data**: download the IBM TabFormer benchmark, rebuild NVIDIA's temporal train/val/test split from the raw CSV with Ray Data, and see in the data why a 0.1% fraud rate dictates how the whole series measures success.